# 02 · A semantic map of the corpus

Input: `embeddings.npy` and `chunks.csv.gz`, produced by `01_embeddings.ipynb`.
Run it in Google Colab, in the same session as notebook 01 (the files are already there),
or upload `embeddings.zip` to the session first.

Steps, following Alammar & Grootendorst, *Hands-On Large Language Models* (2024), ch. 5:

1. **Reduce** the 1,024 dimensions with UMAP: to 2 dimensions to draw the map,
   and to 5 dimensions to find clusters (more dimensions keep more structure than a flat map).
2. **Cluster** the passages with HDBSCAN: dense regions of the 5-D space become "topics";
   passages in no dense region are left as *outliers*.
3. **Describe** each cluster with BERTopic's c-TF-IDF: the words that are frequent in that
   cluster but rare in the others.
4. **Draw** the map, colored by the religious order of the author, by the century of the
   author's death, by the subject of the text (`topic` in the metadata), and by cluster.

UMAP on ~69,000 passages takes a few minutes, so its results are saved and reused
if the notebook is run again in the same session.

## 1. Settings

In [ ]:
!pip install -q bertopic

DATA_DIR = "."                # where embeddings.npy / chunks.csv.gz live and results are saved
FIGURES_DIR = "figures"
METADATA_URL = ("https://raw.githubusercontent.com/vegafabianr/catholic-pastoral-texts/"
                "main/metadatos/textos.csv")
RANDOM_SEED = 42              # UMAP is random; fixing the seed makes the map reproducible
MIN_CLUSTER_SIZE = 150        # the smallest group of passages HDBSCAN will call a cluster

## 2. Load passages, embeddings and metadata

`chunks.csv.gz` and `embeddings.npy` are in the same order: row *i* of one is row *i* of the other.
We attach to every passage the metadata of the text it comes from.

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd

os.makedirs(FIGURES_DIR, exist_ok=True)
if not os.path.exists(f"{DATA_DIR}/embeddings.npy"):      # coming from a fresh session
    with zipfile.ZipFile(f"{DATA_DIR}/embeddings.zip") as z:
        z.extractall(DATA_DIR)

chunks = pd.read_csv(f"{DATA_DIR}/chunks.csv.gz")
embeddings = np.load(f"{DATA_DIR}/embeddings.npy").astype("float32")
texts = pd.read_csv(METADATA_URL)

columns = ["text_id", "title", "author", "author_institution", "author_death_period", "topic"]
chunks = chunks.merge(texts[columns], on="text_id", how="left")
print(f"{len(chunks):,} passages, embeddings of shape {embeddings.shape}")

## 3. Group the small categories

A map cannot show 13 religious orders with 13 distinguishable colors. We keep the seven
orders with most texts and fold the rest (Jerónimos, Benedictinos, laicos...) into "Otros".
The same for subjects: the three largest, plus "Otros".

In [ ]:
ORDERS = ["Jesuita", "Carmelita", "Franciscano", "Dominico",
          "Clero secular", "Concepcionista", "Agustino"]
TOPICS = ["Espiritual", "Historia sacra", "Predicable"]
# The metadata gives the author's death in 50-year periods; we group them into centuries
CENTURY_OF_PERIOD = {"-1500": "Antes de 1500", "1501-1550": "Siglo XVI", "1551-1600": "Siglo XVI",
                     "1601-1650": "Siglo XVII", "1651-1700": "Siglo XVII",
                     "1701-1750": "Siglo XVIII", "1751-": "Siglo XVIII"}
CENTURIES = ["Antes de 1500", "Siglo XVI", "Siglo XVII", "Siglo XVIII"]

order_group = []
for order in chunks["author_institution"]:
    order_group.append(order if order in ORDERS else "Otros")
chunks["order_group"] = order_group

topic_group = []
for topic in chunks["topic"]:
    topic_group.append(topic if topic in TOPICS else "Otros")
chunks["topic_group"] = topic_group

century = []
for period in chunks["author_death_period"]:
    century.append(CENTURY_OF_PERIOD.get(period, "Sin dato"))   # 4 texts have no date of death
chunks["century"] = century

chunks["order_group"].value_counts()

## 4. Reduce dimensions with UMAP

UMAP keeps neighbours together: passages that were close in the 1,024-D space stay close
on the map. Distances between far-away regions mean less than distances inside a region.
`metric="cosine"` matches how the embeddings were built.

In [ ]:
from umap import UMAP

def umap_cached(n_components, min_dist, filename):
    """Run UMAP once and save the result; later runs just load the file."""
    path = f"{DATA_DIR}/{filename}"
    if os.path.exists(path):
        return np.load(path)
    reducer = UMAP(n_components=n_components, n_neighbors=15, min_dist=min_dist,
                   metric="cosine", random_state=RANDOM_SEED, verbose=True)
    result = reducer.fit_transform(embeddings)
    np.save(path, result)
    return result

coords_5d = umap_cached(5, 0.0, "umap_5d.npy")   # for clustering: min_dist=0 packs clusters tightly
coords_2d = umap_cached(2, 0.1, "umap_2d.npy")   # for drawing: a little spread reads better
chunks["x"], chunks["y"] = coords_2d[:, 0], coords_2d[:, 1]

## 5. Clusters and their words (HDBSCAN + BERTopic)

We give BERTopic the 5-D coordinates we already have, so nothing is recomputed:
`BaseDimensionalityReduction` is a "do nothing" step that passes them through.

Three choices matter here, each learned by looking at a first, worse result:

- `cluster_selection_method="leaf"`: HDBSCAN's default ("eom") gave only 8 clusters, one of them
  with 67% of the corpus. "leaf" picks the small dense groups at the tips of HDBSCAN's tree
  (~90 topics). The price: many passages stay as outliers (no topic).
- Proper names out of the labels: otherwise a cluster is described by the people and places of
  one book ("mariquita · pepe · pontevedra"). We drop words written with a capital letter most of
  the time, except religious names, which are the very subject of this corpus.
- `language="multilingual"`: with the default ("english") BERTopic deletes every non-ASCII
  character before counting words, so "españoles" became "espaoles".

`max_df=0.5` drops every word that appears in more than half of the clusters (*que*, *de*, *la*...):
by definition those words cannot tell clusters apart, so no list of stop words is needed.

In [ ]:
from collections import Counter

capitalized, total = Counter(), Counter()
for text in chunks["text"]:
    words = text.split()
    for i, raw in enumerate(words):
        word = raw.strip(".,;:!?()[]«»\"'¿¡-")
        if not word.isalpha():
            continue
        if i > 0 and words[i - 1][-1:] in ".!?":   # skip words that start a sentence
            continue
        total[word.lower()] += 1
        if word[0].isupper():
            capitalized[word.lower()] += 1

# Religious names ARE the topic in this corpus, so we keep them in the labels
KEEP = {"dios", "señor", "cristo", "jesus", "jesucristo", "maria", "virgen", "espiritu", "santo",
        "iglesia", "trinidad", "eucaristia", "sacramento", "evangelio", "padre", "hijo", "madre"}
proper_names = []
for word in total:
    if total[word] >= 3 and capitalized[word] / total[word] > 0.5 and word not in KEEP:
        proper_names.append(word)
print(len(proper_names), "proper names excluded from labels, e.g.", proper_names[:15])

In [ ]:
from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

topic_model = BERTopic(
    language="multilingual",
    umap_model=BaseDimensionalityReduction(),
    hdbscan_model=HDBSCAN(min_cluster_size=100, min_samples=10, metric="euclidean",
                          cluster_selection_method="leaf", prediction_data=True),
    vectorizer_model=CountVectorizer(max_df=0.5, min_df=2, stop_words=proper_names,
                                     token_pattern=r"(?u)\b[a-zñ]{4,}\b"),
)
clusters, _ = topic_model.fit_transform(chunks["text"].str.lower().tolist(), embeddings=coords_5d)
chunks["cluster"] = clusters
print((chunks["cluster"] >= 0).mean().round(2), "of passages in a cluster;",
      len(set(clusters)) - 1, "clusters")

### Readable labels

Keywords are hard to read on a map, so each cluster got a short label written by a language model
after reading its ten keywords, its dominant order and century, and the works most of its
passages come from. These labels are an interpretation: check them against the passages.
The ids below are the ones BERTopic gives (ordered by size); they are stable because HDBSCAN is
deterministic for the same input.

In [ ]:
LABELS = {
    0: "Poesía religiosa", 1: "Maravillas de la creación", 2: "Austeridad de jesuitas ejemplares",
    3: "Coloquios de amor a Dios", 4: "Fundación de colegios jesuitas", 5: "Fundación de monasterios",
    6: "Misión en Japón", 7: "Humildad y vanagloria", 8: "La eucaristía como banquete",
    9: "Misión en Etiopía", 10: "Job: calamidad y desventura", 11: "Cartas familiares (Isla)",
    12: "Tentaciones del demonio", 13: "Vida de la Virgen", 14: "Ágreda a Felipe IV: consejo",
    15: "Misiones entre indígenas", 16: "Confesión y arrepentimiento", 17: "La mujer casada",
    18: "Gracia y virtudes infusas", 19: "El infierno", 20: "Las Moradas",
    21: "Política cristiana", 22: "Muerte y reliquias de santos", 23: "Retórica sagrada",
    24: "Noviciado y estudios", 25: "La misa", 26: "Cartas de Teresa: superiores",
    27: "Pecado y castigo divino", 28: "Fray Gerundio: sátira", 29: "Método de los Ejercicios",
    30: "La fe y el credo", 31: "Ágreda a Felipe IV: gobierno", 32: "Patronazgo nobiliario",
    33: "Mortificación", 34: "Proceso de la Pasión", 35: "Educación de los hijos",
    36: "Novicios y probación", 37: "Murmuración y restitución", 38: "Cartas de Teresa: conventos",
    39: "Dirección espiritual por carta", 40: "Filiación divina", 41: "Mártires y tiranos",
    42: "Paciencia en la adversidad", 43: "Ágreda a Felipe IV: oraciones", 44: "La Anunciación",
    45: "Lágrimas y llanto", 46: "Fugacidad de lo temporal", 47: "Obediencia religiosa",
    48: "Cisma inglés: el divorcio", 49: "Cisma inglés: persecución", 50: "Sacerdocio y órdenes",
    51: "Mártires jesuitas en Inglaterra", 52: "Comunión frecuente", 53: "Bautismo de Cristo",
    54: "Pecados capitales", 55: "Llanto ante la cruz", 56: "Profecías mesiánicas",
    57: "Flagelación y crucifixión", 58: "Oración mental y vocal", 59: "Luis Gonzaga y la nobleza",
    60: "Confesiones de Agustín", 61: "Gobierno de monasterios", 62: "Avaricia",
    63: "María y José", 64: "Muerte incierta, vigilancia", 65: "Ágreda a Felipe IV: guerra",
    66: "Curaciones del Evangelio", 67: "Visiones y revelaciones", 68: "Unción y crisma",
    69: "Idolatría en el Antiguo Testamento", 70: "La lucha con el dragón", 71: "Martirio de santas",
    72: "Teresa: autobiografía", 73: "Blasfemia, juego y mentira", 74: "Ayuno y gula",
    75: "Reliquias que curan", 76: "Misión entre musulmanes", 77: "Naufragios y viajes",
    78: "El Padrenuestro", 79: "Noche oscura", 80: "Cantar de los Cantares",
    81: "Resurrección y apariciones", 82: "Oración de petición", 83: "Prodigios y apariciones",
    84: "Perdón de las ofensas", 85: "Limosna", 86: "Manual de confesión",
    87: "Cántico espiritual", 88: "Constituciones de la Compañía", 89: "Parábola del sembrador",
    90: "El buen pastor", 91: "Fray Gerundio: oratoria",
}

summary = (chunks[chunks["cluster"] >= 0]
           .groupby("cluster")
           .agg(size=("text", "size"),
                order=("order_group", lambda s: s.value_counts().index[0]),
                order_share=("order_group", lambda s: s.value_counts(normalize=True).iloc[0]),
                century=("century", lambda s: s.value_counts().index[0]),
                n_texts=("text_id", "nunique"),
                main_text=("title", lambda s: s.value_counts().index[0]),
                x=("x", "median"), y=("y", "median")))
summary["keywords"] = [" ".join(w for w, _ in topic_model.get_topic(c)[:10]) for c in summary.index]
summary["label"] = summary.index.map(LABELS)
summary.to_csv(f"{DATA_DIR}/clusters.csv")
print(summary.loc[[15, 63, 89], ["label", "keywords"]].to_string())   # check: ñ is back

In [ ]:
summary = (chunks[chunks["cluster"] >= 0]
           .groupby("cluster")
           .agg(size=("text", "size"),
                order=("order_group", lambda s: s.value_counts().index[0]),
                order_share=("order_group", lambda s: s.value_counts(normalize=True).iloc[0]),
                century=("century", lambda s: s.value_counts().index[0]),
                n_texts=("text_id", "nunique"),
                main_text=("title", lambda s: s.value_counts().index[0]),
                x=("x", "median"), y=("y", "median")))
summary["keywords"] = [" ".join(w for w, _ in topic_model.get_topic(c)[:10]) for c in summary.index]
summary["label"] = summary.index.map(LABELS)
summary.to_csv(f"{DATA_DIR}/clusters.csv")
summary.sort_values("size", ascending=False).head(25)

## 6. Drawing style

White background, thin marks, quiet axes (a UMAP map has no meaningful axes, so we hide them).
Colors: the eight categorical slots of a palette checked for color-blind readers. With eight
colors on a scatter some pairs stay close, so every colored map is backed by **direct labels**
and by **small multiples** (one panel per group), where identity never depends on color alone.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "font.family": "DejaVu Sans", "font.size": 10, "text.color": "#0b0b0b",
})
INK, INK_SOFT, BASE_GRAY = "#0b0b0b", "#52514e", "#dddcd8"

SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
ORDER_COLORS = dict(zip(ORDERS, SLOTS))
ORDER_COLORS["Otros"] = "#8f8e89"
TOPIC_COLORS = {"Espiritual": SLOTS[0], "Historia sacra": SLOTS[1], "Predicable": SLOTS[2],
                "Otros": "#8f8e89"}
# Centuries are ordered, so they get one hue (blue) from light (earliest) to dark (latest)
CENTURY_COLORS = dict(zip(CENTURIES, ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]))

def blank_map(ax):
    """Remove axes, ticks and frame: UMAP coordinates have no units."""
    ax.set_xticks([]); ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(False)
    ax.set_aspect("equal")

def halo(width=3):
    """A white outline around text, so labels stay readable on top of points."""
    return [pe.withStroke(linewidth=width, foreground="white")]

def map_by(ax, column, colors, point_size=1.2, alpha=0.35):
    """Gray base of all passages, then each group in its color (largest group drawn first)."""
    ax.scatter(chunks["x"], chunks["y"], s=point_size, c=BASE_GRAY, lw=0, rasterized=True)
    groups = chunks[column].value_counts().index
    for group in groups:
        part = chunks[chunks[column] == group]
        ax.scatter(part["x"], part["y"], s=point_size, c=colors[group], alpha=alpha,
                   lw=0, rasterized=True)
    blank_map(ax)

def label_groups(ax, column, colors, groups):
    """Write each group's name at the median position of its passages (direct labels)."""
    for group in groups:
        part = chunks[chunks[column] == group]
        ax.text(part["x"].median(), part["y"].median(), group, ha="center", va="center",
                fontsize=11, weight="bold", color=INK, path_effects=halo(4))
        ax.scatter(part["x"].median(), part["y"].median() - 0.35, s=40, c=colors[group],
                   edgecolors="white", linewidths=2, zorder=5)

## 7. Map 1 · Religious orders

Every dot is a passage of ~250 words; the larger dots with a white ring are the average
position of each text (the mean of its passages).

In [ ]:
centroids = chunks.groupby("text_id").agg(x=("x", "mean"), y=("y", "mean"),
                                          order_group=("order_group", "first"),
                                          n=("text", "size"))

fig, ax = plt.subplots(figsize=(13, 11))
map_by(ax, "order_group", ORDER_COLORS)
for _, text in centroids.iterrows():
    ax.scatter(text["x"], text["y"], s=18 + text["n"] / 15, c=ORDER_COLORS[text["order_group"]],
               edgecolors="white", linewidths=1.5, zorder=4)
handles = [plt.Line2D([], [], marker="o", ls="", color=ORDER_COLORS[g], label=g, markersize=8)
           for g in ORDERS + ["Otros"]]
ax.legend(handles=handles, frameon=False, loc="lower left", fontsize=10)
ax.set_title("Cartografía semántica del corpus, por orden religiosa del autor",
             loc="left", fontsize=16, weight="bold", color=INK)
fig.text(0.125, 0.06, f"{len(chunks):,} fragmentos de ~250 palabras de 151 textos (bge-m3 + UMAP). "
         "Puntos grandes: posición media de cada texto.", color=INK_SOFT)
fig.savefig(f"{FIGURES_DIR}/01_orders.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Small multiples · one panel per order, one per century

Each panel shows the whole corpus in gray and one group in color. Comparing panels shows
where each order (or century) writes, without having to tell eight colors apart.

In [ ]:
def small_multiples(column, groups, colors, title, filename, ncols=4):
    nrows = -(-len(groups) // ncols)   # ceiling division
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    for ax, group in zip(axes.flat, groups):
        ax.scatter(chunks["x"], chunks["y"], s=0.6, c=BASE_GRAY, lw=0, rasterized=True)
        part = chunks[chunks[column] == group]
        ax.scatter(part["x"], part["y"], s=0.8, c=colors[group], alpha=0.5, lw=0, rasterized=True)
        n_texts = part["text_id"].nunique()
        ax.set_title(f"{group}  ·  {n_texts} textos", loc="left", fontsize=11, weight="bold")
        blank_map(ax)
    for ax in list(axes.flat)[len(groups):]:
        ax.axis("off")
    fig.suptitle(title, x=0.01, ha="left", fontsize=16, weight="bold")
    fig.tight_layout()
    fig.savefig(f"{FIGURES_DIR}/{filename}", dpi=200, bbox_inches="tight")
    plt.show()

small_multiples("order_group", ORDERS + ["Otros"], ORDER_COLORS,
                "Dónde escribe cada orden", "02_orders_small_multiples.png")
small_multiples("century", CENTURIES, CENTURY_COLORS,
                "Dónde escribe cada siglo (siglo de muerte del autor)", "03_centuries_small_multiples.png")

## 9. Map 2 · Subject of the text (`topic`)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
map_by(ax, "topic_group", TOPIC_COLORS)
label_groups(ax, "topic_group", TOPIC_COLORS, TOPICS)
ax.set_title("Cartografía semántica del corpus, por tema del texto",
             loc="left", fontsize=16, weight="bold")
fig.savefig(f"{FIGURES_DIR}/04_topics.png", dpi=200, bbox_inches="tight")
plt.show()

## 10. Map 3 · Topics, colored by their dominant order

Each cluster is drawn in the color of the order that wrote most of its passages; outliers stay gray.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 15))
ax.scatter(chunks["x"], chunks["y"], s=0.6, c=BASE_GRAY, lw=0, rasterized=True)
for cluster_id, row in summary.iterrows():
    part = chunks[chunks["cluster"] == cluster_id]
    ax.scatter(part["x"], part["y"], s=1.2, c=ORDER_COLORS[row["order"]], alpha=0.5,
               lw=0, rasterized=True)
for cluster_id, row in summary.iterrows():
    ax.text(row["x"], row["y"], row["label"], ha="center", va="center", fontsize=7,
            color=INK, path_effects=halo(2.5))
blank_map(ax)
ax.legend(handles=handles, frameon=False, loc="lower left", fontsize=11, markerscale=1.4,
          title="Orden que domina el tópico", title_fontsize=11)
ax.set_title("92 tópicos del corpus, coloreados por la orden que más escribe en cada uno",
             loc="left", fontsize=18, weight="bold")
fig.text(0.125, 0.09, "Tópicos: HDBSCAN sobre UMAP de embeddings bge-m3. Rótulos escritos con IA a partir "
         "de palabras clave (c-TF-IDF) y obras principales. Gris: fragmentos sin tópico.", color=INK_SOFT)
fig.savefig(f"{FIGURES_DIR}/05_clusters.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. Map 4 · One point per text

The passages of a book can be averaged into a single vector: the "center of gravity" of the text.
With only 151 points, UMAP gives a clean map where each author can be named.

In [ ]:
text_ids = sorted(chunks["text_id"].unique())
text_vectors = []
for text_id in text_ids:
    vector = embeddings[(chunks["text_id"] == text_id).to_numpy()].mean(axis=0)
    text_vectors.append(vector / np.linalg.norm(vector))   # back to length 1
text_vectors = np.array(text_vectors)

text_xy = UMAP(n_components=2, n_neighbors=10, min_dist=0.4, metric="cosine",
               random_state=RANDOM_SEED).fit_transform(text_vectors)
by_text = (chunks.groupby("text_id")
           .agg(author=("author", "first"), title=("title", "first"), order=("order_group", "first"),
                century=("century", "first"), n=("text", "size"))
           .loc[text_ids])
by_text["x"], by_text["y"] = text_xy[:, 0], text_xy[:, 1]

def short_author(text_id):
    """Text ids start with the author's id: 'granada-guia-de-pecadores' -> 'Granada'."""
    return text_id.split("-")[0].capitalize()

fig, axes = plt.subplots(1, 2, figsize=(22, 11))
for ax, column, colors, title in [(axes[0], "order", ORDER_COLORS, "Por orden religiosa"),
                                  (axes[1], "century", {**CENTURY_COLORS, "Sin dato": "#c3c2b7"},
                                   "Por siglo de muerte del autor")]:
    ax.scatter(by_text["x"], by_text["y"], s=30 + by_text["n"] / 3,
               c=[colors[v] for v in by_text[column]], edgecolors="white", linewidths=1.2, alpha=0.9)
    for text_id, text in by_text.iterrows():
        ax.text(text["x"], text["y"] + 0.12, short_author(text_id), ha="center", va="bottom",
                fontsize=6.5, color=INK_SOFT, path_effects=halo(2))
    groups = list(colors)
    ax.legend(handles=[plt.Line2D([], [], marker="o", ls="", color=colors[g], label=g, markersize=9)
                       for g in groups if g in set(by_text[column])],
              frameon=False, loc="lower left", fontsize=10)
    ax.set_title(title, loc="left", fontsize=14, weight="bold")
    blank_map(ax)
fig.suptitle("Los 151 textos del corpus en el espacio semántico (tamaño = extensión del texto)",
             x=0.01, ha="left", fontsize=18, weight="bold")
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/06_texts.png", dpi=200, bbox_inches="tight")
plt.show()

## 12. Grouping the topics: four configurations of piety and a double orientation


Each topic is tagged by a language model (from its label, keywords and main works) on two frames:

CONFIGURATION (what kind of piety the topic expresses)
  MC  místico-carismática  - union with God, visions, the soul's inner way (Teresa, Juan de la Cruz)
  AP  ascético-penitencial - discipline, mortification, sin, death and hell, examination of conscience
  DA  devocional-afectiva  - affective meditation (Passion, Virgin, Gospel scenes), love, tears
  BM  benigno-moderada     - ordinary Christian life: family, alms, forgiveness, moderate practice
  OT  otro                 - narrative, institutional or doctrinal (missions, foundations, letters...)

ORIENTATION (what the topic does to the reader)
  D  disciplinar      - orders conduct from outside: rules, threats, duties
  S  subjetivizante   - works from inside: feeling, desire, inner experience
  N  neutra           - neither (mostly the OT topics)

These tags are an interpretation and can be argued with: edit the dictionary and re-run.

In [ ]:
TAGS = {
    0: ("DA", "S"), 1: ("DA", "S"), 2: ("AP", "D"), 3: ("DA", "S"), 4: ("OT", "N"),
    5: ("OT", "N"), 6: ("OT", "N"), 7: ("AP", "D"), 8: ("DA", "S"), 9: ("OT", "N"),
    10: ("AP", "D"), 11: ("OT", "N"), 12: ("AP", "D"), 13: ("MC", "S"), 14: ("OT", "N"),
    15: ("OT", "N"), 16: ("AP", "D"), 17: ("BM", "D"), 18: ("OT", "N"), 19: ("AP", "D"),
    20: ("MC", "S"), 21: ("OT", "N"), 22: ("OT", "N"), 23: ("OT", "N"), 24: ("OT", "N"),
    25: ("DA", "S"), 26: ("OT", "N"), 27: ("AP", "D"), 28: ("OT", "N"), 29: ("AP", "D"),
    30: ("OT", "D"), 31: ("OT", "N"), 32: ("OT", "N"), 33: ("AP", "D"), 34: ("DA", "S"),
    35: ("BM", "D"), 36: ("OT", "N"), 37: ("BM", "D"), 38: ("OT", "N"), 39: ("DA", "S"),
    40: ("DA", "S"), 41: ("OT", "N"), 42: ("AP", "D"), 43: ("OT", "N"), 44: ("DA", "S"),
    45: ("DA", "S"), 46: ("AP", "D"), 47: ("AP", "D"), 48: ("OT", "N"), 49: ("OT", "N"),
    50: ("OT", "D"), 51: ("OT", "N"), 52: ("BM", "S"), 53: ("DA", "S"), 54: ("AP", "D"),
    55: ("DA", "S"), 56: ("OT", "N"), 57: ("DA", "S"), 58: ("DA", "S"), 59: ("OT", "N"),
    60: ("MC", "S"), 61: ("OT", "N"), 62: ("AP", "D"), 63: ("MC", "S"), 64: ("AP", "D"),
    65: ("OT", "N"), 66: ("DA", "S"), 67: ("MC", "S"), 68: ("OT", "D"), 69: ("OT", "N"),
    70: ("MC", "S"), 71: ("OT", "N"), 72: ("MC", "S"), 73: ("AP", "D"), 74: ("AP", "D"),
    75: ("OT", "N"), 76: ("OT", "N"), 77: ("OT", "N"), 78: ("DA", "S"), 79: ("MC", "S"),
    80: ("MC", "S"), 81: ("DA", "S"), 82: ("BM", "S"), 83: ("OT", "N"), 84: ("BM", "D"),
    85: ("BM", "D"), 86: ("AP", "D"), 87: ("MC", "S"), 88: ("OT", "N"), 89: ("DA", "S"),
    90: ("DA", "S"),
    91: ("OT", "N"),
}
CONFIG_NAMES = {"MC": "Místico-carismática", "AP": "Ascético-penitencial",
                "DA": "Devocional-afectiva", "BM": "Benigno-moderada", "OT": "Otro"}
CONFIGS = ["MC", "AP", "DA", "BM"]
# Four hues checked for color-blind readers (all pairs); "Otro" stays gray
CONFIG_COLORS = {"MC": "#6250d6", "AP": "#e34948", "DA": "#eda100", "BM": "#1baf7a", "OT": "#c3c2b7"}

config, orientation = [], []
for cluster_id in chunks["cluster"]:
    tag = TAGS.get(cluster_id, (None, None))   # outliers (-1) get no tag
    config.append(tag[0])
    orientation.append(tag[1])
chunks["config"], chunks["orientation"] = config, orientation
summary["config"] = [TAGS[c][0] for c in summary.index]
summary["orientation"] = [TAGS[c][1] for c in summary.index]
summary.to_csv(f"{DATA_DIR}/clusters.csv")

In [ ]:
# 1. Map: every topic in the color of its configuration
fig, ax = plt.subplots(figsize=(18, 15))
ax.scatter(chunks["x"], chunks["y"], s=0.8, c=BASE_GRAY, lw=0, rasterized=True)
for code in ["OT"] + CONFIGS:
    part = chunks[chunks["config"] == code]
    ax.scatter(part["x"], part["y"], s=2.5, c=CONFIG_COLORS[code], alpha=0.75, lw=0, rasterized=True)
placed = []
for cluster_id, row in summary.sort_values("size", ascending=False).iterrows():
    x, y = row["x"], row["y"]
    half_width = len(row["label"]) * char_width / 2
    moved = True
    while moved:
        moved = False
        for px, py, pw in placed:
            if abs(x - px) < half_width + pw and abs(y - py) < line_height:
                y = py - line_height
                moved = True
    placed.append((x, y, half_width))
    ax.text(x, y, row["label"], ha="center", va="center", fontsize=7.5,
            color=INK if row["config"] != "OT" else INK_SOFT, path_effects=halo(2.5))
blank_map(ax)
ax.legend(handles=[plt.Line2D([], [], marker="o", ls="", color=CONFIG_COLORS[c], label=CONFIG_NAMES[c],
                              markersize=10) for c in CONFIGS + ["OT"]],
          frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=12,
          title="Configuración", title_fontsize=12, alignment="left")
ax.set_title("Cuatro configuraciones de la piedad en el corpus", loc="left", fontsize=20, weight="bold")
fig.text(0.125, 0.09, "Cada tópico (HDBSCAN) fue asignado a una configuración a partir de su rótulo, "
         "palabras clave y obras principales. Gris claro: tópicos narrativos, institucionales o doctrinales; "
         "gris: fragmentos sin tópico.", color=INK_SOFT)
fig.savefig(f"{FIGURES_DIR}/07_configurations_map.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# 2. Composition: of each order's (and each century's) passages in devotional topics,
#    what share falls in each configuration? Horizontal 100% bars, one row per group.
tagged = chunks[chunks["config"].isin(CONFIGS)]

def composition(ax, column, groups, title):
    for row_number, group in enumerate(groups):
        part = tagged[tagged[column] == group]
        left = 0
        for code in CONFIGS:
            share = (part["config"] == code).mean()
            ax.barh(row_number, share, left=left, height=0.7, color=CONFIG_COLORS[code],
                    edgecolor="white", linewidth=2)
            if share >= 0.08:   # write the percentage only where it fits
                ax.text(left + share / 2, row_number, f"{share:.0%}", ha="center", va="center",
                        fontsize=9, color="white", weight="bold")
            left += share
        ax.text(1.01, row_number, f"n={len(part):,}", va="center", fontsize=8, color=INK_SOFT)
    ax.set_yticks(range(len(groups)), groups, fontsize=11)
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xticks([])
    for side in ax.spines.values():
        side.set_visible(False)
    ax.tick_params(left=False)
    ax.set_title(title, loc="left", fontsize=14, weight="bold")

fig, axes = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={"height_ratios": [8, 4]})
composition(axes[0], "order_group", ORDERS + ["Otros"], "Por orden religiosa")
composition(axes[1], "century", CENTURIES, "Por siglo de muerte del autor")
axes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=CONFIG_COLORS[c], label=CONFIG_NAMES[c])
                        for c in CONFIGS],
               frameon=False, ncol=4, loc="lower left", bbox_to_anchor=(0, 1.08), fontsize=11)
fig.suptitle("¿Qué piedad escribe cada orden y cada siglo?", x=0.01, ha="left",
             fontsize=18, weight="bold", y=1.03)
fig.text(0.01, -0.02, "Proporción de fragmentos en tópicos devocionales (se excluyen tópicos "
         "narrativos/institucionales y fragmentos sin tópico). n = fragmentos.", color=INK_SOFT)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/08_configurations_composition.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# 3. Orientation index = (subjetivizante - disciplinar) / (subjetivizante + disciplinar),
#    from -1 (all disciplinary) to +1 (all subjectivizing). One row per order, one dot per century.
oriented = chunks[chunks["orientation"].isin(["D", "S"])]
fig, ax = plt.subplots(figsize=(12, 7))
ax.axvline(0, color="#8f8e89", lw=1, zorder=0)
for row_number, order in enumerate(ORDERS + ["Otros"]):
    ax.axhline(row_number, color="#eeedea", lw=1, zorder=0)
    for century in CENTURIES:
        part = oriented[(oriented["order_group"] == order) & (oriented["century"] == century)]
        if len(part) < 50:   # too few passages to say anything
            continue
        s, d = (part["orientation"] == "S").sum(), (part["orientation"] == "D").sum()
        index = (s - d) / (s + d)
        ax.scatter(index, row_number, s=40 + len(part) / 15, c=CENTURY_COLORS[century],
                   edgecolors="white", linewidths=1.5, zorder=3)
ax.set_yticks(range(len(ORDERS) + 1), ORDERS + ["Otros"], fontsize=11)
ax.invert_yaxis()
ax.set_xlim(-1, 1)
ax.set_xticks([-1, -0.5, 0, 0.5, 1], ["−1\ndisciplinar", "−0,5", "0", "0,5", "+1\nsubjetivizante"])
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.tick_params(left=False)
ax.legend(handles=[plt.Line2D([], [], marker="o", ls="", color=CENTURY_COLORS[c], label=c, markersize=9)
                   for c in CENTURIES],
          frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=10,
          title="Siglo de muerte", title_fontsize=10, alignment="left")
ax.set_title("Doble orientación: disciplinar o subjetivizante, por orden y siglo",
             loc="left", fontsize=16, weight="bold")
fig.text(0.125, -0.03, "Índice = (S − D) / (S + D) sobre los fragmentos en tópicos orientados. "
         "Tamaño del punto = cantidad de fragmentos (mínimo 50).", color=INK_SOFT)
fig.savefig(f"{FIGURES_DIR}/09_orientation.png", dpi=200, bbox_inches="tight")
plt.show()

## 13. Save the passages with their position on the map and their topic

`chunks_mapped.csv.gz` keeps the text of every passage (about 37 MB); `passages.csv.gz` is the same table
without the text, small enough to share with the code. `year` = year of the first edition of the work.

In [ ]:
chunks["year"] = chunks["text_id"].map(texts.set_index("text_id")["first_edition"])
chunks.to_csv(f"{DATA_DIR}/chunks_mapped.csv.gz", index=False)
chunks.drop(columns="text").to_csv(f"{DATA_DIR}/passages.csv.gz", index=False)

## 14. Download the figures and the tables

In [ ]:
!zip -q -r figures.zip figures clusters.csv chunks_mapped.csv.gz passages.csv.gz
from google.colab import files
files.download("figures.zip")